In [1]:
# Initialisers
import sympy as sp
import matplotlib.pyplot as plt
from IPython.display import display
sp.init_printing(use_latex=True)

# Adjoint
def dagger(A):
    return A.conjugate().T

# Kronecker tensor product
def kron(A, B):
    return sp.kronecker_product(A, B)

# Rotation matrix
def rotation(theta):
    return sp.Matrix([[sp.cos(theta), -sp.sin(theta)],
                    [sp.sin(theta),  sp.cos(theta)]])

# Changeable list of theta values
theta_vals = [-sp.pi/2, -sp.pi/4, -sp.pi/8, 0, sp.pi / 8, sp.pi / 4, sp.pi / 2]

In [2]:
# Choose values for the target node, initial position, and the number of time steps
target = int(input("Enter target node: "))
init_pos = int(input("Enter initial position: "))
t = int(input("Enter number of time steps: "))

L = 100
Npos = 2 * L + 1 # Positions -L to L
Ncoin = 2 # Dimension 2-D
dim = Npos * Ncoin
theta = sp.symbols('theta', real=True)

# Map positions to indices
def pos_index(i):
    return i + L

# Shift operator
S = sp.zeros(dim, dim)
for i in range(-L, L + 1):
    if i + 1 <= L:
        S[2 * pos_index(i + 1), 2 * pos_index(i)] = 1 # Move right
    if i - 1 >= -L:
        S[2 * pos_index(i - 1) + 1, 2 * pos_index(i) + 1] = 1 # Move left

# Projection operators
P_pos = sp.zeros(Npos, Npos)
P_pos[pos_index(target), pos_index(target)] = 1
I_coin = sp.eye(2)
P = kron(P_pos, I_coin)
Q = sp.eye(dim) - P

# Initial position
rho_pos = sp.zeros(Npos, Npos)
rho_pos[pos_index(init_pos), pos_index(init_pos)] = 1
rho_coin = sp.Rational(1, 2) * sp.Matrix([[1, 1],
                                        [1, 1]]) # Changeable spin state
rho0 = kron(rho_pos, rho_coin)
R = rotation(theta)
U = S * kron(sp.eye(Npos), R) # Unitary

# Find the expression for p(t)
if init_pos == target:
    if t == 0:
        p_t = 1.0
    else:
        p_t = 0.0
else:
    QU = Q * U
    A = P * U * (QU ** (t - 1))
    B = dagger(A)
    p_t = sp.simplify(sp.trace(A * rho0 * B))
    p_t = sp.trigsimp(p_t)

# Print p(t) symbolically
print("\nSymbolic expression for p(t):")
display(p_t)


Enter target node: 1
Enter initial position: 0
Enter number of time steps: 1

Symbolic expression for p(t):


   2⎛    π⎞
cos ⎜θ + ─⎟
    ⎝    4⎠

In [3]:
dp_t = sp.diff(p_t, theta)
display(dp_t)

      ⎛    π⎞    ⎛    π⎞
-2⋅sin⎜θ + ─⎟⋅cos⎜θ + ─⎟
      ⎝    4⎠    ⎝    4⎠